# FF2: Politische Aufladung der Lebensstil-Cluster

In [1]:
source("setup.R")

data.table 1.17.8 using 8 threads (see ?getDTthreads).  Latest news: r-datatable.com

Attaching package: ‘igraph’

The following objects are masked from ‘package:stats’:

    decompose, spectrum

The following object is masked from ‘package:base’:

    union


Attaching package: ‘dbscan’

The following object is masked from ‘package:stats’:

    as.dendrogram

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.2     ✔ tibble    3.3.0
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.1.0     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ lubridate::%--%()      masks igraph::%--%()
✖ dplyr::as_data_frame() masks tibble::as_data_frame(), igraph::as_data_frame()
✖ dplyr::between()       masks data.table::between()
✖ purrr::compose()       masks igraph::compose()
✖ tidyr::crossing()      masks igraph::crossing()
✖ dplyr::filte

Warning messages:
1: package ‘igraph’ was built under R version 4.5.3 
2: package ‘ineq’ was built under R version 4.5.2 


## Population und Messgröße

In [2]:
df_cluster <- read_csv(CLUSTER_CSV, show_col_types = FALSE)
n_vorher   <- nrow(df_cluster)
df_cluster <- df_cluster |> filter(subreddit %in% lifestyle_alle)

cat(sprintf("df_cluster: %d -> %d Zeilen (Sub x Jahr), entfernt %d\n",
            n_vorher, nrow(df_cluster), n_vorher - nrow(df_cluster)))

fix2016 <- df_cluster |>
  filter(jahr == 2016, cluster != -1) |>
  select(subreddit, cluster)

cat(sprintf("fixe Partition 2016: %d Cluster, %d Subreddits\n",
            n_distinct(fix2016$cluster), nrow(fix2016)))

df_cluster: 196221 -> 196221 Zeilen (Sub x Jahr), entfernt 0
fixe Partition 2016: 95 Cluster, 10941 Subreddits


In [3]:
scores_long <- read_csv(SCORE_CSV, show_col_types = FALSE) |>
  rename(subreddit = 1) |>
  pivot_longer(starts_with("score_"), names_to = "jahr", values_to = "score",
               names_prefix = "score_", names_transform = list(jahr = as.integer))

ref <- scores_long |> filter(jahr == 2016, !is.na(score))
mu0 <- mean(ref$score)
sd0 <- sd(ref$score)
cat(sprintf("z-Referenz 2016 (volles Embedding): n = %d, mu = %.6f, sd = %.6f\n",
            nrow(ref), mu0, sd0))

# Partisan-z eines Jahres, in dieser Form von fast allen Abschnitten gebraucht
scores_jahr <- function(jr) {
  scores_long |>
    filter(jahr == jr) |>
    transmute(subreddit, z = (score - mu0) / sd0) |>
    filter(!is.na(z))
}

dat <- fix2016 |>
  inner_join(scores_long, by = "subreddit") |>
  mutate(z = (score - mu0) / sd0, cluster = factor(cluster)) |>
  filter(!is.na(z))

cat(sprintf("Beobachtungen (Subreddit x Jahr): %d\n", nrow(dat)))

New names:
• `` -> `...1`
z-Referenz 2016 (volles Embedding): n = 16618, mu = 0.005044, sd = 0.067478
Beobachtungen (Subreddit x Jahr): 96869


In [4]:
fixe_subs <- fix2016$subreddit

attrition <- map_dfr(sort(unique(scores_long$jahr)), function(j) {
  hat_score <- fixe_subs %in% scores_jahr(j)$subreddit
  im_raum   <- fixe_subs %in% (df_cluster |> filter(jahr == j) |> pull(subreddit))
  # Die Spalten heissen bewusst anders als die Vektoren: tibble() wertet seine
  # Argumente der Reihe nach aus, eine Spalte namens mit_score wuerde den
  # gleichnamigen Vektor in den Zeilen darunter verdecken.
  tibble(jahr = j,
         mit_score                = sum(hat_score),
         fehlt_gesamt             = sum(!hat_score),
         davon_nicht_im_raum      = sum(!hat_score & !im_raum),
         davon_im_raum_ohne_score = sum(!hat_score & im_raum))
})
print(as.data.frame(attrition))

raus_2024 <- setdiff(fixe_subs, scores_jahr(2024)$subreddit)
dat |>
  filter(jahr == 2016) |>
  mutate(gruppe = if_else(subreddit %in% raus_2024, "bis 2024 ausgeschieden", "bleibt")) |>
  group_by(gruppe) |>
  summarise(n = n(), mean_z = mean(z), sd_z = sd(z), mean_abs_z = mean(abs(z))) |>
  as.data.frame() |>
  print(digits = 3)

  jahr mit_score fehlt_gesamt davon_nicht_im_raum davon_im_raum_ohne_score
1 2016     10941            0                   0                        0
2 2017     10850           91                  91                        0
3 2018     10842           99                  99                        0
4 2019     10820          121                 121                        0
5 2020     10791          150                 150                        0
6 2021     10780          161                 161                        0
7 2022     10754          187                 187                        0
8 2023     10615          326                 326                        0
9 2024     10476          465                 465                        0
                  gruppe     n mean_z  sd_z mean_abs_z
1 bis 2024 ausgeschieden   465 0.1544 0.918      0.732
2                 bleibt 10476 0.0368 0.932      0.739


In [5]:
stich <- df_cluster |>
  group_by(jahr) |>
  summarise(n_gesamt        = n(),
            n_rauschen      = sum(cluster == -1),
            n_geclustert    = n_gesamt - n_rauschen,
            anteil_rauschen = n_rauschen / n_gesamt,
            k_nativ         = n_distinct(cluster[cluster != -1]), .groups = "drop") |>
  left_join(count(dat, jahr, name = "n_voll_sample"), by = "jahr")

print(as.data.frame(stich), digits = 3)
cat(sprintf("\nRauschen %.1f %% (2016) bis %.1f %% | Hauptreihe N %d -> %d\n",
            100 * stich$anteil_rauschen[1], 100 * stich$anteil_rauschen[nrow(stich)],
            stich$n_voll_sample[1], stich$n_voll_sample[nrow(stich)]))

write_csv(stich, file.path(OUT_DIR, "ff2_stichproben.csv"))

  jahr n_gesamt n_rauschen n_geclustert anteil_rauschen k_nativ n_voll_sample
1 2016    15298       4357        10941           0.285      95         10941
2 2017    16860       4575        12285           0.271      95         10850
3 2018    18460       5563        12897           0.301     120         10842
4 2019    20372       5060        15312           0.248     143         10820
5 2020    22492       5845        16647           0.260     165         10791
6 2021    24347       6009        18338           0.247     177         10780
7 2022    25842       6765        19077           0.262     201         10754
8 2023    26563       7013        19550           0.264     212         10615
9 2024    25987       6649        19338           0.256     200         10476

Rauschen 28.5 % (2016) bis 25.6 % | Hauptreihe N 10941 -> 10476


## Eta-Quadrat auf der fixen Partition

In [6]:
cluster_jahr <- dat |>
  group_by(jahr, cluster) |>
  summarise(mean_z = mean(z), var_z = var(z), n = n(), .groups = "drop")

cluster_jahr |>
  group_by(jahr) |>
  summarise(mittlere_abs_ladung = mean(abs(mean_z)), .groups = "drop") |>
  as.data.frame() |>
  print(digits = 4)

  jahr mittlere_abs_ladung
1 2016              0.3816
2 2017              0.4188
3 2018              0.4370
4 2019              0.4707
5 2020              0.4451
6 2021              0.4541
7 2022              0.4583
8 2023              0.4518
9 2024              0.4751


In [7]:
pol_groesse <- cluster_jahr |>
  group_by(jahr) |>
  summarise(k              = n(),
            var_ungew      = var(mean_z),
            var_gew        = sum(n * (mean_z - weighted.mean(mean_z, n))^2) / sum(n),
            mean_abs_ungew = mean(abs(mean_z)),
            mean_abs_gew   = weighted.mean(abs(mean_z), n),
            rho_extrem_n   = suppressWarnings(cor(abs(mean_z), n, method = "spearman")),
            .groups = "drop")

print(as.data.frame(pol_groesse), digits = 3)

kt <- suppressWarnings(cor.test(pol_groesse$jahr, pol_groesse$rho_extrem_n, method = "kendall"))
cat(sprintf("\nrho(|mean_z|, n): %.3f (2016) -> %.3f (2024) | tau = %.3f, p = %.3f\n",
            pol_groesse$rho_extrem_n[1], pol_groesse$rho_extrem_n[nrow(pol_groesse)],
            kt$estimate, kt$p.value))

write_csv(pol_groesse, file.path(OUT_DIR, "ff2_pol_groesse.csv"))

  jahr  k var_ungew var_gew mean_abs_ungew mean_abs_gew rho_extrem_n
1 2016 95     0.247   0.136          0.382        0.266       -0.395
2 2017 95     0.313   0.170          0.419        0.276       -0.283
3 2018 95     0.323   0.174          0.437        0.295       -0.349
4 2019 95     0.348   0.192          0.471        0.322       -0.299
5 2020 95     0.325   0.181          0.445        0.299       -0.394
6 2021 95     0.331   0.200          0.454        0.302       -0.358
7 2022 95     0.344   0.208          0.458        0.322       -0.289
8 2023 95     0.317   0.175          0.452        0.316       -0.410
9 2024 95     0.333   0.215          0.475        0.362       -0.338

rho(|mean_z|, n): -0.395 (2016) -> -0.338 (2024) | tau = -0.056, p = 0.919


In [8]:
set.seed(42)
B <- 999L

kennzahlen <- function(z, cl_idx, n_cl) {
  m  <- as.vector(rowsum(z, cl_idx)) / n_cl
  gm <- sum(n_cl * m) / sum(n_cl)
  c(var_ungew = var(m),
    var_gew   = sum(n_cl * (m - gm)^2) / sum(n_cl),
    abs_ungew = mean(abs(m)),
    abs_gew   = sum(n_cl * abs(m)) / sum(n_cl),
    rho       = suppressWarnings(cor(abs(m), n_cl, method = "spearman")))
}

null_res <- map_dfr(sort(unique(dat$jahr)), function(jr) {
  d      <- filter(dat, jahr == jr)
  cl_idx <- as.integer(factor(d$cluster))
  n_cl   <- tabulate(cl_idx)
  obs    <- kennzahlen(d$z, cl_idx, n_cl)

  sim <- vapply(seq_len(B), function(b) kennzahlen(sample(d$z), cl_idx, n_cl),
                numeric(length(obs)))

  null_mean <- rowMeans(sim)
  # Anteil der Nullziehungen, die mindestens so extrem ausfallen wie der Befund
  p_gr <- (rowSums(sim >= obs) + 1) / (B + 1)
  p_kl <- (rowSums(sim <= obs) + 1) / (B + 1)

  tibble(jahr = jr, k = length(n_cl), n = nrow(d),
         var_gew_obs   = obs["var_gew"],   var_gew_null   = null_mean["var_gew"],
         abs_ungew_obs = obs["abs_ungew"], abs_ungew_null = null_mean["abs_ungew"],
         abs_gew_obs   = obs["abs_gew"],   abs_gew_null   = null_mean["abs_gew"],
         rho_obs       = obs["rho"],       rho_null       = null_mean["rho"],
         rho_p         = p_kl["rho"],      var_gew_p      = p_gr["var_gew"],
         ueberschuss_rho = obs["rho"] - null_mean["rho"],
         verh_obs        = obs["abs_gew"] / obs["abs_ungew"],
         verh_null       = null_mean["abs_gew"] / null_mean["abs_ungew"],
         ueberschuss_verh = obs["abs_gew"] / obs["abs_ungew"] -
                            null_mean["abs_gew"] / null_mean["abs_ungew"])
})

print(as.data.frame(null_res), digits = 3)

for (v in c("ueberschuss_rho", "ueberschuss_verh")) {
  kt <- suppressWarnings(cor.test(null_res$jahr, null_res[[v]], method = "kendall"))
  cat(sprintf("%-17s: %.3f (2016) -> %.3f (2024) | tau = %.3f, p = %.4f\n",
              v, null_res[[v]][1], null_res[[v]][nrow(null_res)], kt$estimate, kt$p.value))
}

write_csv(null_res, file.path(OUT_DIR, "ff2_pol_groesse_null.csv"))

  jahr  k     n var_gew_obs var_gew_null abs_ungew_obs abs_ungew_null
1 2016 95 10941       0.136      0.00748         0.382         0.0977
2 2017 95 10850       0.170      0.00793         0.419         0.0952
3 2018 95 10842       0.174      0.00842         0.437         0.1015
4 2019 95 10820       0.192      0.00841         0.471         0.1044
5 2020 95 10791       0.181      0.00822         0.445         0.1009
6 2021 95 10780       0.200      0.00836         0.454         0.0962
7 2022 95 10754       0.208      0.00865         0.458         0.0982
8 2023 95 10615       0.175      0.00826         0.452         0.1013
9 2024 95 10476       0.215      0.00883         0.475         0.1017
  abs_gew_obs abs_gew_null rho_obs rho_null rho_p var_gew_p ueberschuss_rho
1       0.266       0.0728  -0.395   -0.264 0.076     0.001         -0.1317
2       0.276       0.0653  -0.283   -0.310 0.630     0.001          0.0273
3       0.295       0.0738  -0.349   -0.281 0.229     0.001         -0.0

In [9]:
collapse <- dat |>
  group_by(jahr) |>
  group_modify(~{
    d  <- .x |> mutate(cluster = droplevels(cluster))
    ss <- summary(aov(z ~ cluster, data = d))[[1]][["Sum Sq"]]   # 1 = between, 2 = within
    tibble(eta2 = ss[1] / (ss[1] + ss[2]), between = ss[1], within = ss[2])
  }) |>
  ungroup() |>
  left_join(cluster_jahr |>
              group_by(jahr) |>
              summarise(between_var     = var(mean_z),        # Distinktheit, ungewichtet
                        mean_within_var = mean(var_z, na.rm = TRUE),  # Homogenitaet
                        .groups = "drop"),
            by = "jahr")

print(as.data.frame(collapse), digits = 4)
write_csv(collapse, file.path(OUT_DIR, "ff2_collapse_eta2.csv"))

  jahr   eta2 between within between_var mean_within_var
1 2016 0.1569    1489   8004      0.2469          0.6927
2 2017 0.1853    1841   8094      0.3128          0.7041
3 2018 0.1783    1883   8679      0.3231          0.7323
4 2019 0.1983    2078   8402      0.3477          0.7492
5 2020 0.1923    1950   8191      0.3253          0.6906
6 2021 0.2090    2161   8180      0.3308          0.6903
7 2022 0.2094    2232   8426      0.3442          0.7281
8 2023 0.1880    1862   8038      0.3168          0.6818
9 2024 0.2190    2256   8048      0.3329          0.7304


In [10]:
zerlegung <- collapse |>
  left_join(count(dat, jahr, name = "N"), by = "jahr") |>
  transmute(jahr, N,
            between_gew = between / N,
            within_gew  = within  / N,
            gesamt_var  = between_gew + within_gew,
            eta2_rekonstruiert = between_gew / gesamt_var,
            eta2,
            between_ungew = between_var,
            within_ungew  = mean_within_var)

print(as.data.frame(zerlegung), digits = 4)

for (v in c("between_gew", "within_gew", "between_ungew", "within_ungew")) {
  kt <- suppressWarnings(cor.test(zerlegung$jahr, zerlegung[[v]], method = "kendall"))
  cat(sprintf("%-14s tau = %+.3f, p = %.4f  |  2016: %.4f -> 2024: %.4f\n",
              v, kt$estimate, kt$p.value, zerlegung[[v]][1], zerlegung[[v]][nrow(zerlegung)]))
}

write_csv(zerlegung, file.path(OUT_DIR, "ff2_zerlegung_gewichtet.csv"))

  jahr     N between_gew within_gew gesamt_var eta2_rekonstruiert   eta2
1 2016 10941      0.1361     0.7316     0.8676             0.1569 0.1569
2 2017 10850      0.1696     0.7460     0.9156             0.1853 0.1853
3 2018 10842      0.1737     0.8005     0.9742             0.1783 0.1783
4 2019 10820      0.1921     0.7765     0.9686             0.1983 0.1983
5 2020 10791      0.1807     0.7590     0.9397             0.1923 0.1923
6 2021 10780      0.2005     0.7588     0.9593             0.2090 0.2090
7 2022 10754      0.2076     0.7835     0.9911             0.2094 0.2094
8 2023 10615      0.1754     0.7572     0.9326             0.1880 0.1880
9 2024 10476      0.2154     0.7682     0.9836             0.2190 0.2190
  between_ungew within_ungew
1        0.2469       0.6927
2        0.3128       0.7041
3        0.3231       0.7323
4        0.3477       0.7492
5        0.3253       0.6906
6        0.3308       0.6903
7        0.3442       0.7281
8        0.3168       0.6818
9        

In [11]:
null <- read_csv(file.path(OUT_DIR, "ff2_pol_groesse_null.csv"), show_col_types = FALSE)
zerl <- read_csv(file.path(OUT_DIR, "ff2_zerlegung_gewichtet.csv"), show_col_types = FALSE)

eta2_null <- null |>
  select(jahr, k, n, var_gew_obs, var_gew_null, var_gew_p) |>
  inner_join(select(zerl, jahr, gesamt_var, eta2), by = "jahr") |>
  mutate(eta2_null    = var_gew_null / gesamt_var,
         faktor       = eta2 / eta2_null,
         bench_analyt = (k - 1) / (n - 1))

print(as.data.frame(eta2_null[, c("jahr", "eta2", "eta2_null", "faktor", "var_gew_p")]),
      digits = 4)
write_csv(eta2_null, file.path(OUT_DIR, "ff2_eta2_null_perm.csv"))

  jahr   eta2 eta2_null faktor var_gew_p
1 2016 0.1569  0.008626  18.18     0.001
2 2017 0.1853  0.008657  21.40     0.001
3 2018 0.1783  0.008645  20.63     0.001
4 2019 0.1983  0.008682  22.84     0.001
5 2020 0.1923  0.008750  21.98     0.001
6 2021 0.2090  0.008718  23.97     0.001
7 2022 0.2094  0.008725  24.00     0.001
8 2023 0.1880  0.008858  21.23     0.001
9 2024 0.2190  0.008981  24.38     0.001


## Die native Partition als Gegenprobe

In [12]:
eta2_col <- function(z, g) {
  ok <- !is.na(z); z <- z[ok]; g <- g[ok]
  gm <- mean(z); sst <- sum((z - gm)^2)
  ssb <- sum(tapply(z, g, function(x) length(x) * (mean(x) - gm)^2))
  ssb / sst
}

omega2_col <- function(z, g) {
  ok <- !is.na(z); z <- z[ok]; g <- as.factor(g[ok])
  N <- length(z); k <- nlevels(g); gm <- mean(z)
  sst <- sum((z - gm)^2)
  ssb <- sum(tapply(z, g, function(x) length(x) * (mean(x) - gm)^2))
  ms_w <- (sst - ssb) / (N - k)
  (ssb - (k - 1) * ms_w) / (sst + ms_w)
}

In [13]:
native <- df_cluster |>
  filter(cluster != -1) |>
  transmute(subreddit, jahr = as.integer(jahr), native = cluster)

rep_dat <- fix2016 |>
  rename(fixed = cluster) |>
  inner_join(native, by = "subreddit") |>
  inner_join(scores_long, by = c("subreddit", "jahr")) |>
  mutate(z = (score - mu0) / sd0) |>
  filter(!is.na(z))

repart <- rep_dat |>
  group_by(jahr) |>
  summarise(eta2_fixed    = eta2_col(z, fixed),
            eta2_native   = eta2_col(z, native),
            omega2_fixed  = omega2_col(z, fixed),
            omega2_native = omega2_col(z, native),
            k_fixed  = n_distinct(fixed),
            k_native = n_distinct(native), n = n(), .groups = "drop") |>
  mutate(gap_eta   = eta2_native   - eta2_fixed,
         gap_omega = omega2_native - omega2_fixed,
         # (k-1)/(N-1): so hoch läge eta2 allein durch die Clusterzahl, ohne jedes Signal
         bench_fixed  = (k_fixed  - 1) / (n - 1),
         bench_native = (k_native - 1) / (n - 1))

print(as.data.frame(repart), digits = 3)
write_csv(repart, file.path(OUT_DIR, "ff2_repart_omega2.csv"))

  jahr eta2_fixed eta2_native omega2_fixed omega2_native k_fixed k_native     n
1 2016      0.157       0.157        0.150         0.150      95       95 10941
2 2017      0.200       0.203        0.192         0.195      95       95  9409
3 2018      0.196       0.261        0.187         0.251      94      120  8606
4 2019      0.221       0.294        0.213         0.282      95      142  8934
5 2020      0.211       0.296        0.203         0.283      95      161  8786
6 2021      0.236       0.320        0.228         0.307      95      175  8980
7 2022      0.235       0.329        0.227         0.314      95      195  8754
8 2023      0.223       0.364        0.214         0.348      95      205  8430
9 2024      0.248       0.347        0.239         0.332      95      192  8559
  gap_eta gap_omega bench_fixed bench_native
1 0.00000    0.0000     0.00859      0.00859
2 0.00248    0.0025     0.00999      0.00999
3 0.06571    0.0641     0.01081      0.01383
4 0.07233    0.0693 

In [14]:
bench_tab <- repart |>
  transmute(jahr, k_native, n,
            eta2_native,
            benchmark   = bench_native,
            ueberschuss = eta2_native - bench_native,
            omega2_native)

print(as.data.frame(bench_tab), digits = 3)
cat(sprintf("\nnativ: k %d -> %d | Benchmark %.3f -> %.3f | eta2 %.3f -> %.3f\n",
            bench_tab$k_native[1], bench_tab$k_native[nrow(bench_tab)],
            bench_tab$benchmark[1], bench_tab$benchmark[nrow(bench_tab)],
            bench_tab$eta2_native[1], bench_tab$eta2_native[nrow(bench_tab)]))

write_csv(bench_tab, file.path(OUT_DIR, "ff2_benchmark.csv"))

  jahr k_native     n eta2_native benchmark ueberschuss omega2_native
1 2016       95 10941       0.157   0.00859       0.148         0.150
2 2017       95  9409       0.203   0.00999       0.193         0.195
3 2018      120  8606       0.261   0.01383       0.248         0.251
4 2019      142  8934       0.294   0.01578       0.278         0.282
5 2020      161  8786       0.296   0.01821       0.278         0.283
6 2021      175  8980       0.320   0.01938       0.301         0.307
7 2022      195  8754       0.329   0.02216       0.307         0.314
8 2023      205  8430       0.364   0.02420       0.340         0.348
9 2024      192  8559       0.347   0.02232       0.325         0.332

nativ: k 95 -> 192 | Benchmark 0.009 -> 0.022 | eta2 0.157 -> 0.347


In [15]:
native_frei <- native |>
  inner_join(scores_long, by = c("subreddit", "jahr")) |>
  mutate(z = (score - mu0) / sd0) |>
  filter(!is.na(z)) |>
  group_by(jahr) |>
  summarise(eta2_native_frei   = eta2_col(z, native),
            omega2_native_frei = omega2_col(z, native),
            k_native = n_distinct(native), n = n(),
            bench = (n_distinct(native) - 1) / (n() - 1), .groups = "drop") |>
  mutate(ueberschuss = eta2_native_frei - bench)

print(as.data.frame(native_frei), digits = 3)

for (v in c("eta2_native_frei", "omega2_native_frei")) {
  kt <- suppressWarnings(cor.test(native_frei$jahr, native_frei[[v]], method = "kendall"))
  cat(sprintf("%-19s: %.3f -> %.3f | tau = %.3f, p = %.4f\n", v,
              native_frei[[v]][1], native_frei[[v]][nrow(native_frei)],
              kt$estimate, kt$p.value))
}
write_csv(native_frei, file.path(OUT_DIR, "ff2_native_frei.csv"))

native_ls <- native |>
  filter(subreddit %in% lifestyle_alle) |>
  inner_join(scores_long, by = c("subreddit", "jahr")) |>
  mutate(z = (score - mu0) / sd0) |>
  filter(!is.na(z)) |>
  group_by(jahr) |>
  summarise(eta2_native_ls   = eta2_col(z, native),
            omega2_native_ls = omega2_col(z, native),
            k_native = n_distinct(native), n = n(),
            bench    = (n_distinct(native) - 1) / (n() - 1), .groups = "drop")

print(as.data.frame(native_ls))
write_csv(native_ls, file.path(OUT_DIR, "ff2_native_frei_lifestyle.csv"))

# Kontrolle statt Gegenprobe: die Clusterdatei enthaelt bereits nur
# Lebensstil-Subreddits, der Filter in der Populationszelle entfernt null
# Zeilen. Der zweite Filter auf lifestyle_alle kann deshalb nichts mehr
# wegnehmen, beide Reihen muessen exakt uebereinstimmen. Geprueft wird diese
# Invariante und nicht die Robustheit gegen den Ausschluss der Etiketten.
cat("\nKontrolle: der Lebensstil-Filter greift auf der nativen Partition nicht mehr.\n")
cat(sprintf("  max |Delta| eta2 = %.2e | max |Delta| omega2 = %.2e | max |Delta| n = %d\n",
            max(abs(native_frei$eta2_native_frei   - native_ls$eta2_native_ls)),
            max(abs(native_frei$omega2_native_frei - native_ls$omega2_native_ls)),
            max(abs(native_frei$n - native_ls$n))))

  jahr eta2_native_frei omega2_native_frei k_native     n   bench ueberschuss
1 2016            0.157              0.150       95 10941 0.00859       0.148
2 2017            0.196              0.190       95 12285 0.00765       0.189
3 2018            0.232              0.225      120 12897 0.00923       0.223
4 2019            0.257              0.250      143 15312 0.00927       0.248
5 2020            0.260              0.253      165 16647 0.00985       0.250
6 2021            0.282              0.275      177 18338 0.00960       0.272
7 2022            0.302              0.294      201 19077 0.01048       0.291
8 2023            0.331              0.324      212 19550 0.01079       0.321
9 2024            0.351              0.345      200 19338 0.01029       0.341
eta2_native_frei   : 0.157 -> 0.351 | tau = 1.000, p = 0.0000
omega2_native_frei : 0.150 -> 0.345 | tau = 1.000, p = 0.0000
  jahr eta2_native_ls omega2_native_ls k_native     n       bench
1 2016      0.1568537        0

## Sortierung ohne Partition

In [16]:
K_MORAN <- 50L

# L2-normalisiert; fängt die Leerspalte ab, die durch ein Leerzeichen am
# Zeilenende entsteht
lese_vektoren <- function(pfad) {
  df <- read_delim(pfad, delim = " ", skip = 1, col_names = FALSE, show_col_types = FALSE)
  df <- df |> select(where(~ !all(is.na(.x))))
  M  <- as.matrix(df[, -1])
  rownames(M) <- df[[1]]
  M / sqrt(rowSums(M^2))
}

# Moran auf einem kNN-Graphen. Bei fester Nachbarzahl sind binäre und
# zeilenstandardisierte Gewichte äquivalent.
moran_knn <- function(z, id) {
  kk <- ncol(id)
  zc <- z - mean(z)
  nb_sum <- rowSums(matrix(zc[id], nrow = nrow(id)))
  N <- length(z); W <- N * kk
  (N / W) * sum(zc * nb_sum) / sum(zc^2)
}

# Kantenliste {i, j : cos > tau, j > i}, blockweise, damit die volle
# Ähnlichkeitsmatrix nie im Speicher steht
threshold_edges <- function(M, tau, block = 2000) {
  n <- nrow(M)
  out <- list()
  for (a in seq(1, n, by = block)) {
    b <- min(a + block - 1, n)
    S <- M[a:b, , drop = FALSE] %*% t(M)
    idx <- which(S > tau, arr.ind = TRUE)
    if (nrow(idx) == 0) next
    zeile  <- (a:b)[idx[, 1]]
    spalte <- idx[, 2]
    keep <- spalte > zeile
    if (!any(keep)) next
    out[[length(out) + 1]] <- data.table(
      from = zeile[keep], to = spalte[keep], weight = S[idx[keep, , drop = FALSE]])
  }
  el <- rbindlist(out)
  el[, `:=`(from = rownames(M)[from], to = rownames(M)[to])]
  el[]
}

# Die kNN-Suche über die volle Landschaft ist teuer und wird mehrfach gebraucht,
# deshalb einmal rechnen und auf Platte legen. Cache verwerfen, wenn sich
# Vektoren oder Scores ändern.
prep_knn <- function(jahr, k, subs = NULL, cache_dir = "cache_knn") {
  dir.create(cache_dir, showWarnings = FALSE, recursive = TRUE)
  f <- file.path(cache_dir, sprintf("knn_%d_k%d.rds", jahr, k))
  if (file.exists(f)) return(readRDS(f))
  V   <- lese_vektoren(ali_datei(jahr))
  sc  <- scores_jahr(jahr)
  sub <- intersect(rownames(V), sc$subreddit)
  if (!is.null(subs)) sub <- intersect(rownames(V), intersect(subs, sc$subreddit))
  res <- list(sub = sub, id = dbscan::kNN(V[sub, , drop = FALSE], k = k)$id, jahr = jahr)
  saveRDS(res, f)
  res
}

In [17]:
# Population: volle gescorte Landschaft samt der politischen Subreddits,
# also eine andere als bei eta2. rel_sd mittelt ausserdem nur über die
# Subreddits der fixen Partition, moran_I über alle Knoten.
distinkt_jahr <- map_dfr(JAHRE, function(j) {
  pk  <- prep_knn(j, K_MORAN)
  sub <- pk$sub; id <- pk$id
  sc  <- scores_jahr(j)
  z   <- sc$z[match(sub, sc$subreddit)]
  global_sd <- sd(z, na.rm = TRUE)

  moran <- moran_knn(z, id)

  sd_nb  <- apply(id, 1, function(ix) sd(z[ix], na.rm = TRUE))
  fokus  <- sub %in% fix2016$subreddit
  rel_sd <- mean(sd_nb[fokus], na.rm = TRUE) / global_sd

  tibble(jahr = j, moran_I = moran, rel_sd = rel_sd, global_sd = global_sd, n = length(z))
})

print(as.data.frame(distinkt_jahr), digits = 4)

for (v in c("moran_I", "rel_sd")) {
  kt <- suppressWarnings(cor.test(distinkt_jahr$jahr, distinkt_jahr[[v]], method = "kendall"))
  cat(sprintf("%-8s tau = %+.3f, p = %.4f  |  2016: %.3f -> 2024: %.3f\n",
              v, kt$estimate, kt$p.value,
              distinkt_jahr[[v]][1], distinkt_jahr[[v]][nrow(distinkt_jahr)]))
}

write_csv(distinkt_jahr, file.path(OUT_DIR, "ff2_moran_relsd.csv"))

  jahr moran_I rel_sd global_sd     n
1 2016  0.3360 0.7454     1.000 16618
2 2017  0.3525 0.7652     1.003 18426
3 2018  0.3950 0.7766     1.006 20255
4 2019  0.4027 0.7519     1.013 22409
5 2020  0.4422 0.7473     1.008 24842
6 2021  0.4576 0.7408     1.007 26930
7 2022  0.4854 0.7202     1.045 28640
8 2023  0.4835 0.7179     1.017 29446
9 2024  0.5058 0.7122     1.038 28848
moran_I  tau = +0.944, p = 0.0000  |  2016: 0.336 -> 2024: 0.506
rel_sd   tau = -0.722, p = 0.0059  |  2016: 0.745 -> 2024: 0.712


In [18]:
library(spdep)

moran_inf <- map_dfr(JAHRE, function(j) {
  pk  <- prep_knn(j, K_MORAN)
  sub <- pk$sub; id <- pk$id
  sc  <- scores_jahr(j)
  z   <- sc$z[match(sub, sc$subreddit)]

  knn_obj <- structure(list(nn = id, np = nrow(id), k = K_MORAN,
                            dimension = NA_integer_, x = NULL), class = "knn")
  lw <- spdep::nb2listw(spdep::knn2nb(knn_obj, sym = FALSE), style = "W", zero.policy = TRUE)

  mt <- spdep::moran.test(z, lw, zero.policy = TRUE)
  mc <- spdep::moran.mc(z, lw, nsim = 999, zero.policy = TRUE)

  tibble(jahr = j, n = length(z),
         moran_I  = unname(mt$estimate[1]),
         E_I      = unname(mt$estimate[2]),
         sd_I     = sqrt(unname(mt$estimate[3])),
         z_score  = unname(mt$statistic),
         p_analyt = mt$p.value,
         p_perm   = mc$p.value)
})

print(as.data.frame(moran_inf), digits = 4)
write_csv(moran_inf, file.path(OUT_DIR, "ff2_moran_spdep.csv"))

Loading required package: spData
To access larger datasets in this package, install the spDataLarge
package with: `install.packages('spDataLarge',
repos='https://nowosad.github.io/drat/', type='source')`
Loading required package: sf
Linking to GEOS 3.14.1, GDAL 3.12.1, PROJ 9.7.1; sf_use_s2() is TRUE


Warning messages:
1: package ‘spdep’ was built under R version 4.5.3 
2: package ‘spData’ was built under R version 4.5.3 
3: package ‘sf’ was built under R version 4.5.3 


  jahr     n moran_I        E_I     sd_I z_score p_analyt p_perm
1 2016 16618  0.3360 -6.018e-05 0.001322   254.2        0  0.001
2 2017 18426  0.3525 -5.427e-05 0.001268   278.0        0  0.001
3 2018 20255  0.3950 -4.937e-05 0.001226   322.3        0  0.001
4 2019 22409  0.4027 -4.463e-05 0.001174   343.2        0  0.001
5 2020 24842  0.4422 -4.026e-05 0.001121   394.4        0  0.001
6 2021 26930  0.4576 -3.713e-05 0.001079   424.3        0  0.001
7 2022 28640  0.4854 -3.492e-05 0.001044   464.7        0  0.001
8 2023 29446  0.4835 -3.396e-05 0.001028   470.2        0  0.001
9 2024 28848  0.5058 -3.467e-05 0.001041   485.9        0  0.001


In [19]:
K_SET <- c(15, 50, 100)

moran_k <- map_dfr(JAHRE, function(j) {
  V   <- lese_vektoren(ali_datei(j))
  sc  <- scores_jahr(j)
  sub <- intersect(rownames(V), sc$subreddit)
  V   <- V[sub, , drop = FALSE]
  z   <- sc$z[match(sub, sc$subreddit)]
  map_dfr(K_SET, function(kk) {
    tibble(jahr = j, k = kk, moran_I = moran_knn(z, dbscan::kNN(V, k = kk)$id), n = length(z))
  })
})

print(as.data.frame(moran_k), digits = 3)
write_csv(moran_k, file.path(OUT_DIR, "ff2_moran_ksweep.csv"))

   jahr   k moran_I     n
1  2016  15   0.425 16618
2  2016  50   0.336 16618
3  2016 100   0.282 16618
4  2017  15   0.443 18426
5  2017  50   0.352 18426
6  2017 100   0.295 18426
7  2018  15   0.483 20255
8  2018  50   0.395 20255
9  2018 100   0.337 20255
10 2019  15   0.502 22409
11 2019  50   0.403 22409
12 2019 100   0.343 22409
13 2020  15   0.541 24842
14 2020  50   0.442 24842
15 2020 100   0.379 24842
16 2021  15   0.552 26930
17 2021  50   0.458 26930
18 2021 100   0.396 26930
19 2022  15   0.583 28640
20 2022  50   0.485 28640
21 2022 100   0.421 28640
22 2023  15   0.574 29446
23 2023  50   0.484 29446
24 2023 100   0.426 29446
25 2024  15   0.602 28848
26 2024  50   0.506 28848
27 2024 100   0.440 28848


In [20]:
moran_panel <- map_dfr(JAHRE, function(j) {
  V   <- lese_vektoren(ali_datei(j))
  sc  <- scores_jahr(j)
  sub <- intersect(rownames(V), intersect(fix2016$subreddit, sc$subreddit))
  z   <- sc$z[match(sub, sc$subreddit)]
  id  <- dbscan::kNN(V[sub, , drop = FALSE], k = K_MORAN)$id
  tibble(jahr = j, k = K_MORAN, moran_I = moran_knn(z, id), n = length(sub))
})

print(as.data.frame(moran_panel), digits = 4)
kt <- suppressWarnings(cor.test(moran_panel$jahr, moran_panel$moran_I, method = "kendall"))
cat(sprintf("\nMoran auf konstanter Population: tau = %+.3f, p = %.4f | %.3f -> %.3f | n %d -> %d\n",
            kt$estimate, kt$p.value,
            moran_panel$moran_I[1], moran_panel$moran_I[nrow(moran_panel)],
            moran_panel$n[1], moran_panel$n[nrow(moran_panel)]))

write_csv(moran_panel, file.path(OUT_DIR, "ff2_moran_panel.csv"))

  jahr  k moran_I     n
1 2016 50  0.3007 10941
2 2017 50  0.3178 10850
3 2018 50  0.3680 10842
4 2019 50  0.3658 10820
5 2020 50  0.3799 10791
6 2021 50  0.4071 10780
7 2022 50  0.4170 10754
8 2023 50  0.4089 10615
9 2024 50  0.4054 10476

Moran auf konstanter Population: tau = +0.722, p = 0.0059 | 0.301 -> 0.405 | n 10941 -> 10476


In [21]:
subs_2016 <- rownames(lese_vektoren(ali_datei(2016)))
cat("Subreddits im 2016er Vektorraum:", length(subs_2016), "\n")

moran_lokal <- map_dfr(JAHRE, function(j) {
  V   <- lese_vektoren(ali_datei(j))
  sc  <- scores_jahr(j)
  sub <- intersect(rownames(V), intersect(lifestyle_alle, sc$subreddit))
  z   <- sc$z[match(sub, sc$subreddit)]
  id  <- dbscan::kNN(V[sub, , drop = FALSE], k = K_MORAN)$id

  zc  <- z - mean(z)
  lag <- rowMeans(matrix(zc[id], nrow = nrow(id)))
  Ii  <- zc * lag / mean(zc^2)

  alt <- sub %in% subs_2016

  tibble(jahr = j,
         moran_global = mean(Ii),
         moran_alt    = mean(Ii[alt]),
         moran_neu    = if (any(!alt)) mean(Ii[!alt]) else NA_real_,
         anteil_alt   = mean(alt),
         n_alt = sum(alt), n_neu = sum(!alt), n = length(z))
})

print(as.data.frame(moran_lokal), digits = 4)

for (v in c("moran_global", "moran_alt", "moran_neu", "anteil_alt")) {
  d  <- moran_lokal |> filter(is.finite(.data[[v]]))
  kt <- suppressWarnings(cor.test(d$jahr, d[[v]], method = "kendall"))
  cat(sprintf("%-13s tau = %+.3f, p = %.4f  |  %d: %.3f -> %d: %.3f\n",
              v, kt$estimate, kt$p.value, d$jahr[1], d[[v]][1],
              d$jahr[nrow(d)], d[[v]][nrow(d)]))
}

write_csv(moran_lokal, file.path(OUT_DIR, "ff2_moran_lokal.csv"))

Subreddits im 2016er Vektorraum: 16618 
  jahr moran_global moran_alt moran_neu anteil_alt n_alt n_neu     n
1 2016       0.3045    0.3045        NA     1.0000 15298     0 15298
2 2017       0.3233    0.3233    0.3237     0.8987 15152  1708 16860
3 2018       0.3713    0.3803    0.3303     0.8206 15149  3311 18460
4 2019       0.3731    0.3879    0.3302     0.7425 15127  5245 20372
5 2020       0.4031    0.4226    0.3633     0.6709 15091  7401 22492
6 2021       0.4277    0.4339    0.4176     0.6192 15076  9271 24347
7 2022       0.4579    0.4380    0.4858     0.5823 15049 10793 25842
8 2023       0.4539    0.4439    0.4667     0.5607 14894 11669 26563
9 2024       0.4767    0.4425    0.5214     0.5665 14721 11266 25987
moran_global  tau = +0.944, p = 0.0000  |  2016: 0.305 -> 2024: 0.477
moran_alt     tau = +0.944, p = 0.0000  |  2016: 0.305 -> 2024: 0.443
moran_neu     tau = +0.857, p = 0.0017  |  2017: 0.324 -> 2024: 0.521
anteil_alt    tau = -0.944, p = 0.0000  |  2016: 1.000 -> 20

In [22]:
knoten <- map(JAHRE, function(j) {
  namen <- fread(ali_datei(j), select = 1, skip = 1, header = FALSE)[[1]]
  intersect(intersect(namen, lifestyle_alle), scores_jahr(j)$subreddit)
})
names(knoten) <- as.character(JAHRE)

eintritt <- tibble(subreddit = unlist(knoten),
                   jahr      = rep(JAHRE, lengths(knoten))) |>
  group_by(subreddit) |>
  summarise(eintrittsjahr = min(jahr), .groups = "drop")

cat("Kohortengroessen:\n")
print(as.data.frame(count(eintritt, eintrittsjahr)))

kohorten <- map_dfr(JAHRE[-1], function(j) {
  sub <- knoten[[as.character(j)]]
  neu <- intersect(eintritt |> filter(eintrittsjahr == j) |> pull(subreddit), sub)
  if (length(neu) < 30) return(NULL)

  V  <- lese_vektoren(ali_datei(j))[sub, , drop = FALSE]
  sc <- scores_jahr(j)
  z  <- sc$z[match(sub, sc$subreddit)]
  zc <- z - mean(z); m2 <- mean(zc^2)     # Zentrierung und Skala aus dem ganzen Graphen

  # Nachbarn nur für die Kohorte suchen, aber im vollen Graphen. k+1, weil der
  # Abfragepunkt selbst als nächster Nachbar zurückkommt.
  qi <- match(neu, sub)
  nn <- dbscan::kNN(V, k = K_MORAN + 1L, query = V[qi, , drop = FALSE])$id
  id <- t(vapply(seq_len(nrow(nn)), function(r) {
    v <- nn[r, ]; head(v[v != qi[r]], K_MORAN)
  }, integer(K_MORAN)))

  Ii_neu <- zc[qi] * rowMeans(matrix(zc[id], nrow = nrow(id))) / m2

  tibble(kohorte = j, n_kohorte = length(neu), n_graph = length(sub),
         moran_kohorte = mean(Ii_neu),
         anteil_neu_am_graph = length(neu) / length(sub))
})

kohorten <- kohorten |>
  left_join(select(moran_lokal, kohorte = jahr, moran_alt, moran_global), by = "kohorte")

print(as.data.frame(kohorten), digits = 4)
kt <- suppressWarnings(cor.test(kohorten$kohorte, kohorten$moran_kohorte, method = "kendall"))
cat(sprintf("\nKohorten-Trend: tau = %+.3f, p = %.4f  |  %d: %.3f -> %d: %.3f\n",
            kt$estimate, kt$p.value,
            kohorten$kohorte[1], kohorten$moran_kohorte[1],
            kohorten$kohorte[nrow(kohorten)], kohorten$moran_kohorte[nrow(kohorten)]))

write_csv(kohorten, file.path(OUT_DIR, "ff2_moran_kohorten.csv"))

Kohortengroessen:
  eintrittsjahr     n
1          2016 15298
2          2017  1708
3          2018  1668
4          2019  1951
5          2020  2212
6          2021  1905
7          2022  1614
8          2023  1242
9          2024   331
  kohorte n_kohorte n_graph moran_kohorte anteil_neu_am_graph moran_alt
1    2017      1708   16860        0.3237             0.10130    0.3233
2    2018      1668   18460        0.3168             0.09036    0.3803
3    2019      1951   20372        0.2961             0.09577    0.3879
4    2020      2212   22492        0.3240             0.09835    0.4226
5    2021      1905   24347        0.3893             0.07824    0.4339
6    2022      1614   25842        0.3997             0.06246    0.4380
7    2023      1242   26563        0.3725             0.04676    0.4439
8    2024       331   25987        0.4612             0.01274    0.4425
  moran_global
1       0.3233
2       0.3713
3       0.3731
4       0.4031
5       0.4277
6       0.4579
7       0

In [23]:
moran_ls_inf <- map_dfr(JAHRE, function(j) {
  pk  <- prep_knn(j, K_MORAN, subs = lifestyle_alle, cache_dir = "cache_knn_lifestyle")
  sub <- pk$sub; id <- pk$id
  sc  <- scores_jahr(j)
  z   <- sc$z[match(sub, sc$subreddit)]

  knn_obj <- structure(list(nn = id, np = nrow(id), k = K_MORAN,
                            dimension = NA_integer_, x = NULL), class = "knn")
  lw <- spdep::nb2listw(spdep::knn2nb(knn_obj, sym = FALSE), style = "W", zero.policy = TRUE)

  mt <- spdep::moran.test(z, lw, zero.policy = TRUE)
  mc <- spdep::moran.mc(z, lw, nsim = 999, zero.policy = TRUE)

  tibble(jahr = j, n = length(z),
         moran_I  = unname(mt$estimate[1]),
         E_I      = unname(mt$estimate[2]),
         sd_I     = sqrt(unname(mt$estimate[3])),
         moran_z_test = unname(mt$statistic),
         p_analyt = mt$p.value,
         p_perm   = mc$p.value)
})

print(as.data.frame(moran_ls_inf), digits = 4)
cat(sprintf("Gegencheck gegen die Zerlegung: max |Delta| = %.2e\n",
            max(abs(moran_ls_inf$moran_I - moran_lokal$moran_global))))

write_csv(moran_ls_inf, file.path(OUT_DIR, "ff2_moran_lifestyle_inf.csv"))

  jahr     n moran_I        E_I     sd_I moran_z_test p_analyt p_perm
1 2016 15298  0.3045 -6.537e-05 0.001376        221.3        0  0.001
2 2017 16860  0.3233 -5.932e-05 0.001325        244.0        0  0.001
3 2018 18460  0.3713 -5.417e-05 0.001284        289.2        0  0.001
4 2019 20372  0.3731 -4.909e-05 0.001231        303.0        0  0.001
5 2020 22492  0.4031 -4.446e-05 0.001179        342.0        0  0.001
6 2021 24347  0.4277 -4.107e-05 0.001135        376.8        0  0.001
7 2022 25842  0.4579 -3.870e-05 0.001101        416.0        0  0.001
8 2023 26563  0.4539 -3.765e-05 0.001085        418.5        0  0.001
9 2024 25987  0.4767 -3.848e-05 0.001098        434.0        0  0.001
Gegencheck gegen die Zerlegung: max |Delta| = 6.72e-15


In [24]:
moran_beide <- function(el, sub, zc) {
  N <- length(zc); E <- nrow(el); nenner <- sum(zc^2)
  fi <- match(el$from, sub); ti <- match(el$to, sub)

  I_bin <- (N / E) * sum(zc[fi] * zc[ti]) / nenner

  deg <- tabulate(c(fi, ti), nbins = N)
  agg <- rowsum(c(zc[ti], zc[fi]), group = c(fi, ti), reorder = FALSE)
  acc <- numeric(N); acc[as.integer(rownames(agg))] <- agg[, 1]
  lag <- ifelse(deg > 0, acc / deg, 0)
  # Kein Vorfaktor N/(N - n_iso): spdep reduziert bei style "W" sowohl n als auch
  # die Gewichtssumme um die isolierten Knoten, der Vorfaktor ist damit eins.
  list(I_w = sum(zc * lag) / nenner, I_bin = I_bin,
       n_iso = sum(deg == 0), mean_deg = 2 * E / N)
}

TAUS <- c(0.55, 0.60, 0.65, 0.70)

thr_ls <- map_dfr(JAHRE, function(j) {
  V   <- lese_vektoren(ali_datei(j))
  sc  <- scores_jahr(j)
  sub <- intersect(rownames(V), intersect(lifestyle_alle, sc$subreddit))
  M   <- V[sub, , drop = FALSE]
  M   <- M / sqrt(rowSums(M^2))
  z   <- sc$z[match(sub, sc$subreddit)]
  zc  <- z - mean(z); N <- length(zc)

  map_dfr(TAUS, function(tt) {
    message("jahr = ", j, "  tau = ", tt)
    el <- threshold_edges(M, tt)
    if (nrow(el) == 0)
      return(tibble(jahr = j, tau = tt, n = N, n_edges = 0L, n_iso = N,
                    mean_deg = 0, moran_I = NA_real_, moran_I_binaer = NA_real_))
    r <- moran_beide(el, sub, zc)
    tibble(jahr = j, tau = tt, n = N, n_edges = nrow(el), n_iso = r$n_iso,
           mean_deg = r$mean_deg, moran_I = r$I_w, moran_I_binaer = r$I_bin)
  })
})

print(as.data.frame(thr_ls), digits = 4)

write_csv(thr_ls, file.path(OUT_DIR, "ff2_moran_threshold_lifestyle.csv"))

trend_ls <- map_dfr(TAUS, function(tt) {
  d   <- thr_ls |> filter(tau == tt) |> arrange(jahr)
  kt  <- cor.test(d$jahr, d$moran_I,        method = "kendall")
  ktb <- cor.test(d$jahr, d$moran_I_binaer, method = "kendall")
  bind_rows(
    tibble(mass  = sprintf("moran_threshold_ls_tau%03d", round(tt * 100)),
           tau   = round(unname(kt$estimate), 3), p = signif(kt$p.value, 3),
           start = d$moran_I[1], ende = d$moran_I[nrow(d)], n_jahre = nrow(d)),
    tibble(mass  = sprintf("moran_threshold_ls_tau%03d_bin", round(tt * 100)),
           tau   = round(unname(ktb$estimate), 3), p = signif(ktb$p.value, 3),
           start = d$moran_I_binaer[1], ende = d$moran_I_binaer[nrow(d)], n_jahre = nrow(d)))
})

print(as.data.frame(trend_ls), digits = 4)
write_csv(trend_ls, file.path(OUT_DIR, "ff2_moran_threshold_lifestyle_trends.csv"))

jahr = 2016  tau = 0.55
jahr = 2016  tau = 0.6
jahr = 2016  tau = 0.65
jahr = 2016  tau = 0.7
jahr = 2017  tau = 0.55
jahr = 2017  tau = 0.6
jahr = 2017  tau = 0.65
jahr = 2017  tau = 0.7
jahr = 2018  tau = 0.55
jahr = 2018  tau = 0.6
jahr = 2018  tau = 0.65
jahr = 2018  tau = 0.7
jahr = 2019  tau = 0.55
jahr = 2019  tau = 0.6
jahr = 2019  tau = 0.65
jahr = 2019  tau = 0.7
jahr = 2020  tau = 0.55
jahr = 2020  tau = 0.6
jahr = 2020  tau = 0.65
jahr = 2020  tau = 0.7
jahr = 2021  tau = 0.55
jahr = 2021  tau = 0.6
jahr = 2021  tau = 0.65
jahr = 2021  tau = 0.7
jahr = 2022  tau = 0.55
jahr = 2022  tau = 0.6
jahr = 2022  tau = 0.65
jahr = 2022  tau = 0.7
jahr = 2023  tau = 0.55
jahr = 2023  tau = 0.6
jahr = 2023  tau = 0.65
jahr = 2023  tau = 0.7
jahr = 2024  tau = 0.55
jahr = 2024  tau = 0.6
jahr = 2024  tau = 0.65
jahr = 2024  tau = 0.7
   jahr  tau     n n_edges n_iso mean_deg moran_I moran_I_binaer
1  2016 0.55 15298 1177213   641   153.90  0.3487         0.1858
2  2016 0.60 15298  7080

In [25]:
moran_k_ls <- map_dfr(JAHRE, function(j) {
  V   <- lese_vektoren(ali_datei(j))
  sc  <- scores_jahr(j)
  sub <- intersect(rownames(V), intersect(lifestyle_alle, sc$subreddit))
  V   <- V[sub, , drop = FALSE]
  z   <- sc$z[match(sub, sc$subreddit)]
  map_dfr(K_SET, function(kk) {
    tibble(jahr = j, k = kk, moran_I = moran_knn(z, dbscan::kNN(V, k = kk)$id), n = length(z))
  })
})

print(as.data.frame(moran_k_ls), digits = 4)

write_csv(moran_k_ls, file.path(OUT_DIR, "ff2_moran_ksweep_lifestyle.csv"))

trend_k_ls <- moran_k_ls |>
  group_by(k) |>
  group_modify(~{
    kt <- suppressWarnings(cor.test(.x$jahr, .x$moran_I, method = "kendall"))
    tibble(tau = round(unname(kt$estimate), 3), p = signif(kt$p.value, 3),
           start = .x$moran_I[which.min(.x$jahr)],
           ende  = .x$moran_I[which.max(.x$jahr)], n_jahre = nrow(.x))
  }) |>
  ungroup() |>
  transmute(mass = sprintf("moran_ls_k%d", k), tau, p, start, ende, n_jahre)

print(as.data.frame(trend_k_ls), digits = 4)
write_csv(trend_k_ls, file.path(OUT_DIR, "ff2_moran_ksweep_lifestyle_trends.csv"))

   jahr   k moran_I     n
1  2016  15  0.3790 15298
2  2016  50  0.3045 15298
3  2016 100  0.2584 15298
4  2017  15  0.4087 16860
5  2017  50  0.3233 16860
6  2017 100  0.2710 16860
7  2018  15  0.4550 18460
8  2018  50  0.3713 18460
9  2018 100  0.3166 18460
10 2019  15  0.4726 20372
11 2019  50  0.3731 20372
12 2019 100  0.3157 20372
13 2020  15  0.5051 22492
14 2020  50  0.4031 22492
15 2020 100  0.3410 22492
16 2021  15  0.5210 24347
17 2021  50  0.4277 24347
18 2021 100  0.3670 24347
19 2022  15  0.5593 25842
20 2022  50  0.4579 25842
21 2022 100  0.3946 25842
22 2023  15  0.5472 26563
23 2023  50  0.4539 26563
24 2023 100  0.3965 26563
25 2024  15  0.5761 25987
26 2024  50  0.4767 25987
27 2024 100  0.4106 25987
           mass   tau        p  start   ende n_jahre
1  moran_ls_k15 0.944 4.96e-05 0.3790 0.5761       9
2  moran_ls_k50 0.944 4.96e-05 0.3045 0.4767       9
3 moran_ls_k100 0.944 4.96e-05 0.2584 0.4106       9


In [26]:
diag <- thr_ls |>
  mutate(iso_anteil = n_iso / n) |>
  select(jahr, tau, n, n_iso, iso_anteil, mean_deg, moran_I) |>
  arrange(tau, jahr)

diag |>
  filter(jahr %in% c(2016, 2024)) |>
  select(tau, jahr, iso_anteil, mean_deg, moran_I) |>
  pivot_wider(names_from = jahr, values_from = c(iso_anteil, mean_deg, moran_I)) |>
  as.data.frame() |>
  print(digits = 3)

diag |> filter(tau == 0.70) |> as.data.frame() |> print(digits = 3)

   tau iso_anteil_2016 iso_anteil_2024 mean_deg_2016 mean_deg_2024 moran_I_2016
1 0.55          0.0419          0.0173         153.9         212.3        0.349
2 0.60          0.1288          0.0395          92.6         116.8        0.366
3 0.65          0.2529          0.0758          53.3          60.8        0.346
4 0.70          0.3974          0.1342          27.7          29.0        0.309
  moran_I_2024
1        0.436
2        0.498
3        0.538
4        0.557
  jahr tau     n n_iso iso_anteil mean_deg moran_I
1 2016 0.7 15298  6079      0.397     27.7   0.309
2 2017 0.7 16860  6278      0.372     19.4   0.351
3 2018 0.7 18460  6034      0.327     14.6   0.394
4 2019 0.7 20372  5106      0.251     17.9   0.439
5 2020 0.7 22492  4389      0.195     17.0   0.490
6 2021 0.7 24347  4228      0.174     18.4   0.501
7 2022 0.7 25842  4071      0.158     20.7   0.534
8 2023 0.7 26563  3735      0.141     25.1   0.531
9 2024 0.7 25987  3487      0.134     29.0   0.557


In [27]:
TAU_CORE <- 0.70

lokal_jahr <- function(j) {
  V   <- lese_vektoren(ali_datei(j))
  sc  <- scores_jahr(j)
  sub <- intersect(rownames(V), intersect(lifestyle_alle, sc$subreddit))
  M   <- V[sub, , drop = FALSE]; M <- M / sqrt(rowSums(M^2))
  z   <- sc$z[match(sub, sc$subreddit)]
  zc  <- z - mean(z); N <- length(zc)

  el  <- threshold_edges(M, TAU_CORE)
  fi <- match(el$from, sub); ti <- match(el$to, sub)
  deg <- tabulate(c(fi, ti), nbins = N)
  agg <- rowsum(c(zc[ti], zc[fi]), group = c(fi, ti), reorder = FALSE)
  acc <- numeric(N); acc[as.integer(rownames(agg))] <- agg[, 1]
  lag <- ifelse(deg > 0, acc / deg, 0)

  tibble(jahr = j, subreddit = sub, I_i = zc * lag / (sum(zc^2) / N), verbunden = deg > 0)
}

lok <- map_dfr(JAHRE, lokal_jahr)

core <- lok |>
  filter(verbunden) |>
  count(subreddit) |>
  filter(n == length(JAHRE)) |>
  pull(subreddit)

res_core <- lok |>
  group_by(jahr) |>
  summarise(n_full = n(), I_full = mean(I_i),
            n_core = sum(subreddit %in% core),
            I_core = mean(I_i[subreddit %in% core]), .groups = "drop")

print(as.data.frame(res_core), digits = 4)

ktf <- cor.test(res_core$jahr, res_core$I_full, method = "kendall")
ktc <- cor.test(res_core$jahr, res_core$I_core, method = "kendall")
cat(sprintf(paste0("\ncos >= %.2f\n",
                   "  alle Knoten     : %.3f -> %.3f  (tau = %+.3f, p = %.3g)\n",
                   "  immer verbunden : %.3f -> %.3f  (tau = %+.3f, p = %.3g), n = %d\n"),
            TAU_CORE,
            res_core$I_full[1], res_core$I_full[nrow(res_core)],
            unname(ktf$estimate), ktf$p.value,
            res_core$I_core[1], res_core$I_core[nrow(res_core)],
            unname(ktc$estimate), ktc$p.value, length(core)))

  jahr n_full I_full n_core I_core
1 2016  15298 0.3088   6705 0.5453
2 2017  16860 0.3507   6705 0.5755
3 2018  18460 0.3938   6705 0.6103
4 2019  20372 0.4392   6705 0.6013
5 2020  22492 0.4896   6705 0.6335
6 2021  24347 0.5008   6705 0.5863
7 2022  25842 0.5342   6705 0.6039
8 2023  26563 0.5310   6705 0.5960
9 2024  25987 0.5566   6705 0.6001

cos >= 0.70
  alle Knoten     : 0.309 -> 0.557  (tau = +0.944, p = 4.96e-05)
  immer verbunden : 0.545 -> 0.600  (tau = +0.222, p = 0.477), n = 6705


In [28]:
RAEUME <- c("nativ", "aligned")

vek_raum <- function(jahr, raum) {
  if (raum == "nativ" || jahr == 2016) vek_datei(jahr) else ali_datei(jahr)
}

load_norm <- function(jahr, subs, raum) {
  v <- fread(vek_raum(jahr, raum), skip = 1, header = FALSE)
  setnames(v, 1, "sub")
  v <- v[sub %in% subs]
  M <- as.matrix(v[, -1])
  M <- M[, colSums(is.na(M)) < nrow(M), drop = FALSE]
  rownames(M) <- v$sub
  M / sqrt(rowSums(M^2))
}

# Kantenliste in eine spdep-Nachbarschaft. Isolierte Knoten werden als 0L
# kodiert, das ist die Konvention von spdep.
edges_to_nb <- function(el, knoten) {
  n   <- length(knoten)
  idx <- setNames(seq_len(n), knoten)
  fi  <- idx[el$from]; ti <- idx[el$to]
  nb  <- split(c(ti, fi), factor(c(fi, ti), levels = seq_len(n)))
  nb  <- lapply(nb, function(x) if (length(x) == 0) 0L else sort(unique(as.integer(x))))
  names(nb) <- NULL; class(nb) <- "nb"; attr(nb, "region.id") <- knoten
  nb
}

moran_schwelle <- function(jahr, tau, raum, basis) {
  sc   <- scores_jahr(jahr)
  subs <- intersect(basis, sc$subreddit)
  M    <- load_norm(jahr, subs, raum)
  subs <- rownames(M)
  el   <- threshold_edges(M, tau)
  z    <- sc$z[match(subs, sc$subreddit)]
  nb   <- edges_to_nb(el, subs)
  lw   <- spdep::nb2listw(nb, style = "W", zero.policy = TRUE)
  mt   <- spdep::moran.test(z, lw, zero.policy = TRUE)
  tibble(raum = raum, jahr = jahr, tau = tau, n = length(z), n_edges = nrow(el),
         n_iso = sum(spdep::card(nb) == 0), mean_deg = 2 * nrow(el) / length(z),
         moran_I = unname(mt$estimate[1]), E_I = unname(mt$estimate[2]),
         sd_I = sqrt(unname(mt$estimate[3])), z_score = unname(mt$statistic),
         p_analyt = mt$p.value)
}

sweep_population <- function(voll) {
  map_dfr(RAEUME, function(rm)
    map_dfr(TAUS, function(tt)
      map_dfr(JAHRE, function(j) {
        message("raum = ", rm, "  tau = ", tt, "  jahr = ", j,
                "  population = ", if (voll) "voll" else "panel")
        basis <- if (voll) scores_jahr(j)$subreddit else ana_subs
        moran_schwelle(j, tt, rm, basis)
      })))
}

for (voll in c(TRUE, FALSE)) {
  sweep_thr <- sweep_population(voll)

  trend_thr <- sweep_thr |>
    group_by(raum, tau) |>
    summarise(moran_16 = moran_I[which.min(jahr)], moran_24 = moran_I[which.max(jahr)],
              tau_Moran = cor(jahr, moran_I, method = "kendall"),
              p = suppressWarnings(cor.test(jahr, moran_I, method = "kendall")$p.value),
              iso_16 = n_iso[which.min(jahr)], iso_24 = n_iso[which.max(jahr)],
              mean_deg_16 = round(mean_deg[which.min(jahr)], 1), .groups = "drop")

  paritaet <- sweep_thr |>
    select(raum, jahr, tau, moran_I) |>
    pivot_wider(names_from = raum, values_from = moran_I) |>
    mutate(diff = abs(nativ - aligned))

  cat("\n== Population:", if (voll) "volle gescorte Landschaft" else "Lebensstil-Panel", "==\n")
  print(as.data.frame(trend_thr), digits = 3)
  cat(sprintf("Paritaet nativ gegen ausgerichtet: max |Delta| = %.4f, Median = %.4f\n",
              max(paritaet$diff, na.rm = TRUE), median(paritaet$diff, na.rm = TRUE)))

  write_csv(sweep_thr, file.path(OUT_DIR, sprintf("ff2_moran_threshold_sweep%s.csv",
                                                  if (voll) "_voll" else "")))
}

raum = nativ  tau = 0.55  jahr = 2016  population = voll
raum = nativ  tau = 0.55  jahr = 2017  population = voll
raum = nativ  tau = 0.55  jahr = 2018  population = voll
raum = nativ  tau = 0.55  jahr = 2019  population = voll
raum = nativ  tau = 0.55  jahr = 2020  population = voll
raum = nativ  tau = 0.55  jahr = 2021  population = voll
raum = nativ  tau = 0.55  jahr = 2022  population = voll
raum = nativ  tau = 0.55  jahr = 2023  population = voll
raum = nativ  tau = 0.55  jahr = 2024  population = voll
raum = nativ  tau = 0.6  jahr = 2016  population = voll
raum = nativ  tau = 0.6  jahr = 2017  population = voll
raum = nativ  tau = 0.6  jahr = 2018  population = voll
raum = nativ  tau = 0.6  jahr = 2019  population = voll
raum = nativ  tau = 0.6  jahr = 2020  population = voll
raum = nativ  tau = 0.6  jahr = 2021  population = voll
raum = nativ  tau = 0.6  jahr = 2022  population = voll
raum = nativ  tau = 0.6  jahr = 2023  population = voll
raum = nativ  tau = 0.6  jahr = 2024  p

## Die Delle 2023

In [29]:
# Die zehn Seed-Paare der politischen Achse. Bei mehreren expliziten Paaren ist
# die Achse der auf Länge eins normierte Mittelwert der Differenzvektoren
# rechts minus links.
SEEDS <- list(
  c("Liberal", "Conservative"),
  c("progressive", "conservatives"),
  c("Democrat", "Republican"),
  c("Political_Revolution", "ConservativesOnly"),
  c("AskALiberal", "askaconservative"),
  c("AskDemocrats", "AskTrumpSupporters"),
  c("askhillarysupporters", "AskThe_Donald"),
  c("hillaryclinton", "The_Donald"),
  c("SandersForPresident", "HillaryForPrison"),
  c("Impeach_Trump", "HillaryMeltdown"))

achse_aus_seeds <- function(M) {
  da <- Filter(function(p) all(p %in% rownames(M)), SEEDS)
  d  <- M[vapply(da, function(p) p[2], ""), , drop = FALSE] -
        M[vapply(da, function(p) p[1], ""), , drop = FALSE]
  v  <- colMeans(d)
  list(achse = v / sqrt(sum(v^2)), n_paare = length(da),
       fehlend = Filter(function(p) !all(p %in% rownames(M)), SEEDS))
}

raum_normiert <- function(pfad) {
  v <- fread(pfad, skip = 1, header = FALSE)
  setnames(v, 1, "sub")
  M <- as.matrix(v[, -1])
  M <- M[, colSums(is.na(M)) < nrow(M), drop = FALSE]
  rownames(M) <- v$sub
  M / sqrt(rowSums(M^2))
}

M16     <- raum_normiert(vek_datei(2016))
achse16 <- achse_aus_seeds(M16)
cat(sprintf("2016: Achse aus %d von %d Seed-Paaren\n", achse16$n_paare, length(SEEDS)))
if (length(achse16$fehlend))
  cat("  fehlende Paare:",
      paste(vapply(achse16$fehlend, paste, "", collapse = "/"), collapse = ", "), "\n")

alle_scores <- read.csv(file.path(PROJ, "Data/Vektoren/all_scores.csv"), row.names = 1)

delle <- map_dfr(JAHRE, function(j) {
  Mj <- if (j == 2016) M16 else raum_normiert(ali_datei(j))
  aj <- achse_aus_seeds(Mj)

  ank_pfad <- file.path(PROJ, sprintf("final_anker_16_%d_full.csv", j))
  ank <- if (j != 2016 && file.exists(ank_pfad)) read_csv(ank_pfad, show_col_types = FALSE) else NULL

  sp <- paste0("partisan_", j)
  tibble(jahr = j,
         seed_paare = aj$n_paare,
         cos_achse_zu_2016  = sum(aj$achse * achse16$achse),
         anker_n            = if (is.null(ank)) NA_real_ else nrow(ank),
         anker_score_mean   = if (is.null(ank)) NA_real_ else mean(ank$score),
         anker_score_median = if (is.null(ank)) NA_real_ else median(ank$score),
         anker_score_min    = if (is.null(ank)) NA_real_ else min(ank$score),
         score_sd_roh       = if (sp %in% names(alle_scores))
                                sd(alle_scores[[sp]], na.rm = TRUE) else NA_real_)
})

print(as.data.frame(delle), digits = 6)
write_csv(delle, file.path(OUT_DIR, "ff2_delle_alignment.csv"))

d   <- delle |> filter(jahr %in% c(2022, 2023, 2024))
ref <- mean(d$cos_achse_zu_2016[d$jahr != 2023])
cat(sprintf("\n2023 gegen den Mittelwert aus 2022 und 2024: %+.2f %% beim Kosinus zur 2016er Achse.\n",
            100 * (d$cos_achse_zu_2016[d$jahr == 2023] - ref) / ref))
cat("Deutlich negativ hiesse Rotationsartefakt, nahe null schliesst das Alignment aus.\n")

2016: Achse aus 10 von 10 Seed-Paaren
  jahr seed_paare cos_achse_zu_2016 anker_n anker_score_mean anker_score_median
1 2016         10          1.000000      NA               NA                 NA
2 2017         10          0.364338    2368         0.751744           0.738573
3 2018          9          0.407432    2346         0.734944           0.722081
4 2019          9          0.432502    2327         0.732586           0.722212
5 2020          9          0.336978    2346         0.751279           0.739790
6 2021          8          0.285016    2313         0.763700           0.751621
7 2022          8          0.267017    2300         0.736787           0.724053
8 2023          8          0.311765    2269         0.739567           0.725787
9 2024          8          0.393964    2252         0.735847           0.721891
  anker_score_min score_sd_roh
1              NA    0.0674784
2        0.685954    0.0677084
3        0.673303    0.0679003
4        0.668826    0.0683234
5      

In [30]:
R_RICHTUNGEN <- 200L
set.seed(20260723)

raeume <- lapply(JAHRE, function(j) raum_normiert(ali_datei(j)))
names(raeume) <- as.character(JAHRE)
panel_ali <- Reduce(intersect, lapply(raeume, rownames))
DIM <- ncol(raeume[[1]])
cat("Panel über alle neun Jahresräume:", length(panel_ali), "Subreddits |",
    DIM, "Dimensionen\n")

achse_pol <- achse_aus_seeds(raeume[["2016"]])$achse

# Einmal gezogen und für alle Jahre dieselben, sonst mischt sich
# Ziehungsrauschen in den Jahresvergleich.
Q <- matrix(rnorm(DIM * R_RICHTUNGEN), nrow = DIM)
Q <- Q / rep(sqrt(colSums(Q^2)), each = DIM)
cat(sprintf("Bei perfekt isotroper Wolke waere die Streuung 1/sqrt(%d) = %.4f\n",
            DIM, 1 / sqrt(DIM)))

placebo <- map_dfr(c("jahr", "panel"), function(stichprobe) {
  map_dfr(JAHRE, function(j) {
    M <- raeume[[as.character(j)]]
    V <- if (stichprobe == "jahr") M else M[panel_ali, , drop = FALSE]
    sd_pol  <- sd(as.vector(V %*% achse_pol))
    sd_plac <- apply(V %*% Q, 2, sd)
    tibble(stichprobe = stichprobe, jahr = j, n_subs = nrow(V),
           sd_partisan       = sd_pol,
           sd_placebo_mean   = mean(sd_plac),
           sd_placebo_median = median(sd_plac),
           sd_placebo_p05    = unname(quantile(sd_plac, 0.05)),
           sd_placebo_p95    = unname(quantile(sd_plac, 0.95)),
           verhaeltnis       = sd_pol / mean(sd_plac))
  })
})

print(as.data.frame(placebo), digits = 5)
write_csv(placebo, file.path(OUT_DIR, "ff2_placebo_achse.csv"))

cat("\n2023 gegen den Mittelwert aus 2022 und 2024, in Prozent:\n")
for (st in c("jahr", "panel")) {
  dd  <- placebo |> filter(stichprobe == st)
  aus <- vapply(c("sd_partisan", "sd_placebo_mean", "verhaeltnis"), function(sp) {
    r <- mean(dd[[sp]][dd$jahr %in% c(2022, 2024)])
    sprintf("%+7.2f %%", 100 * (dd[[sp]][dd$jahr == 2023] - r) / r)
  }, character(1))
  cat(sprintf("  %-6s sd_partisan %s | sd_placebo %s | Verhaeltnis %s\n",
              st, aus[1], aus[2], aus[3]))
}
cat("Faellt die Placebo-Streuung 2023 aehnlich stark und bleibt das Verhaeltnis flach,\n",
    "ist die Delle eine globale Kompression des Raumes und nichts politisch Spezifisches.\n")

rm(raeume); invisible(gc(verbose = FALSE))

Panel über alle neun Jahresräume: 15522 Subreddits | 150 Dimensionen
Bei perfekt isotroper Wolke waere die Streuung 1/sqrt(150) = 0.0816
   stichprobe jahr n_subs sd_partisan sd_placebo_mean sd_placebo_median
1        jahr 2016  16618    0.067478        0.070229          0.070045
2        jahr 2017  18426    0.067708        0.070493          0.070273
3        jahr 2018  20255    0.067900        0.070664          0.070445
4        jahr 2019  22409    0.068323        0.070775          0.070537
5        jahr 2020  24842    0.068011        0.070600          0.070406
6        jahr 2021  26930    0.067947        0.070648          0.070119
7        jahr 2022  28640    0.070509        0.070704          0.070216
8        jahr 2023  29446    0.068601        0.070877          0.070292
9        jahr 2024  28848    0.070064        0.070769          0.070635
10      panel 2016  15522    0.067026        0.070482          0.070307
11      panel 2017  15522    0.068070        0.071015          0.070832

## Trägt der Befund nur die gekippten Subreddits?

In [31]:
llm <- fread(LLM_CSV, colClasses = "character", header = TRUE)
llm[, subreddit := sub("^r/", "", subreddit)]
llm <- unique(llm, by = "subreddit")

M_lab   <- as.matrix(llm[, as.character(JAHRE), with = FALSE])
spaeter <- M_lab[, as.character(2017:2024), drop = FALSE]
n_true  <- rowSums(spaeter == "true", na.rm = TRUE)
n_obs   <- rowSums(!is.na(spaeter) & spaeter != "")
letztes <- apply(M_lab, 1, function(r) {
  r <- r[!is.na(r) & r != ""]; if (!length(r)) NA_character_ else r[length(r)]
})
ist_ls <- llm$subreddit %in% lifestyle_alle

kipp_any  <- llm$subreddit[ist_ls & n_true >= 1]
kipp_mehr <- llm$subreddit[ist_ls & n_obs > 0 & n_true / pmax(n_obs, 1) >= 0.5]
kipp_end  <- llm$subreddit[ist_ls & !is.na(letztes) & letztes == "true"]

cat(sprintf("Lebensstil-Subs %d | Kipper any %d (%.1f %%), mehrheitlich %d (%.1f %%), Endzustand %d (%.1f %%)\n",
            length(lifestyle_alle),
            length(kipp_any),  100 * length(kipp_any)  / length(lifestyle_alle),
            length(kipp_mehr), 100 * length(kipp_mehr) / length(lifestyle_alle),
            length(kipp_end),  100 * length(kipp_end)  / length(lifestyle_alle)))

eta2_von <- function(d) {
  gm <- mean(d$z)
  ssb <- d |> group_by(cluster) |>
    summarise(n = n(), m = mean(z), .groups = "drop") |>
    summarise(s = sum(n * (m - gm)^2)) |> pull(s)
  ssb / sum((d$z - gm)^2)
}

kipper_eta2 <- map_dfr(sort(unique(dat$jahr)), function(jr) {
  d <- filter(dat, jahr == jr)
  tibble(jahr        = jr,
         eta2_voll   = eta2_von(d),
         eta2_o_any  = eta2_von(filter(d, !subreddit %in% kipp_any)),
         eta2_o_mehr = eta2_von(filter(d, !subreddit %in% kipp_mehr)),
         eta2_o_end  = eta2_von(filter(d, !subreddit %in% kipp_end)),
         n_voll      = nrow(d),
         n_o_any     = sum(!d$subreddit %in% kipp_any))
})

print(as.data.frame(kipper_eta2), digits = 4)

moran_global_von <- function(V, z) {
  id  <- dbscan::kNN(V, k = K_MORAN)$id
  zc  <- z - mean(z)
  mean(zc * rowMeans(matrix(zc[id], nrow = nrow(id))) / mean(zc^2))
}

kipper_moran <- map_dfr(JAHRE, function(j) {
  V   <- lese_vektoren(ali_datei(j))
  sc  <- scores_jahr(j)
  sub <- intersect(rownames(V), intersect(lifestyle_alle, sc$subreddit))
  Vs  <- V[sub, , drop = FALSE]
  z   <- sc$z[match(sub, sc$subreddit)]

  # (i) unveränderter Graph, lokales Moran nach Kipper-Status zerlegt
  id  <- dbscan::kNN(Vs, k = K_MORAN)$id
  zc  <- z - mean(z)
  Ii  <- zc * rowMeans(matrix(zc[id], nrow = nrow(id))) / mean(zc^2)
  kip <- sub %in% kipp_any

  # (ii) Graph ohne die Kipper neu gebaut, sie sind dann auch als Nachbarn weg
  I_neu <- if (sum(!kip) > K_MORAN + 1)
    moran_global_von(Vs[!kip, , drop = FALSE], z[!kip]) else NA_real_

  tibble(jahr = j,
         moran_voll    = mean(Ii),
         moran_kipper  = if (any(kip))  mean(Ii[kip])  else NA_real_,
         moran_rest    = if (any(!kip)) mean(Ii[!kip]) else NA_real_,
         anteil_kipper = mean(kip),
         moran_ohne_kipper_neu = I_neu,
         n = length(z), n_kipper = sum(kip))
})

print(as.data.frame(kipper_moran), digits = 4)

trend_von <- function(d, v) {
  d  <- filter(d, is.finite(.data[[v]]))
  kt <- suppressWarnings(cor.test(d$jahr, d[[v]], method = "kendall"))
  tibble(reihe = v, start = d[[v]][1], ende = d[[v]][nrow(d)],
         delta = d[[v]][nrow(d)] - d[[v]][1],
         tau = unname(kt$estimate), p = kt$p.value,
         j_start = d$jahr[1], j_ende = d$jahr[nrow(d)])
}

kipper_trends <- bind_rows(
  map_dfr(c("eta2_voll", "eta2_o_any", "eta2_o_mehr", "eta2_o_end"),
          ~ trend_von(kipper_eta2, .x)),
  map_dfr(c("moran_voll", "moran_rest", "moran_kipper", "moran_ohne_kipper_neu", "anteil_kipper"),
          ~ trend_von(kipper_moran, .x)))

print(as.data.frame(kipper_trends), digits = 4)

d_voll <- kipper_trends$delta[kipper_trends$reihe == "moran_voll"]
d_ohne <- kipper_trends$delta[kipper_trends$reihe == "moran_ohne_kipper_neu"]
e_voll <- kipper_trends$delta[kipper_trends$reihe == "eta2_voll"]
e_ohne <- kipper_trends$delta[kipper_trends$reihe == "eta2_o_any"]
cat(sprintf("\nMoran-Anstieg voll %+.3f, ohne Kipper %+.3f -> %.0f %% bleiben\n",
            d_voll, d_ohne, 100 * d_ohne / d_voll))
cat(sprintf("eta2-Anstieg  voll %+.3f, ohne Kipper %+.3f -> %.0f %% bleiben\n",
            e_voll, e_ohne, 100 * e_ohne / e_voll))

write_csv(kipper_eta2,   file.path(OUT_DIR, "ff2_kipper_eta2.csv"))
write_csv(kipper_moran,  file.path(OUT_DIR, "ff2_kipper_check.csv"))
write_csv(kipper_trends, file.path(OUT_DIR, "ff2_kipper_check_trends.csv"))

Lebensstil-Subs 27929 | Kipper any 1568 (5.6 %), mehrheitlich 441 (1.6 %), Endzustand 727 (2.6 %)
  jahr eta2_voll eta2_o_any eta2_o_mehr eta2_o_end n_voll n_o_any
1 2016    0.1569     0.1462      0.1540     0.1527  10941    9818
2 2017    0.1853     0.1739      0.1825     0.1809  10850    9729
3 2018    0.1783     0.1740      0.1757     0.1774  10842    9725
4 2019    0.1983     0.1926      0.1975     0.1958  10820    9702
5 2020    0.1923     0.1683      0.1858     0.1812  10791    9673
6 2021    0.2090     0.2023      0.2062     0.2043  10780    9665
7 2022    0.2094     0.2010      0.2086     0.2060  10754    9639
8 2023    0.1880     0.1845      0.1868     0.1848  10615    9505
9 2024    0.2190     0.2033      0.2165     0.2128  10476    9375
  jahr moran_voll moran_kipper moran_rest anteil_kipper moran_ohne_kipper_neu
1 2016     0.3045       0.4299     0.2902       0.10237                0.2905
2 2017     0.3233       0.4455     0.3109       0.09259                0.3103
3 2018  

## Trends und Herkunft der Etiketten

In [32]:
# Stichprobe je Zeile: fix = Voll-Sample, nativ = Schnittmenge beider
# Partitionen, nativ_lifestyle = Population der Moran-Reihe, Haupt-Moran
# = volle gescorte Landschaft.
tau_trend <- function(jahr, wert) {
  jahr <- as.numeric(jahr); wert <- as.numeric(wert)
  ok <- is.finite(jahr) & is.finite(wert); jahr <- jahr[ok]; wert <- wert[ok]
  o  <- order(jahr)
  kt <- suppressWarnings(cor.test(jahr, wert, method = "kendall"))
  data.frame(tau = round(unname(kt$estimate), 3), p = signif(kt$p.value, 3),
             start = wert[o][1], ende = wert[o][length(wert)], n_jahre = length(jahr))
}

lies <- function(name) read.csv(file.path(OUT_DIR, name))
co <- lies("ff2_collapse_eta2.csv");        re <- lies("ff2_repart_omega2.csv")
mo <- lies("ff2_moran_spdep.csv");          rs <- lies("ff2_moran_relsd.csv")
ze <- lies("ff2_zerlegung_gewichtet.csv");  mk <- lies("ff2_moran_ksweep.csv")
mp <- lies("ff2_moran_panel.csv");          nf <- lies("ff2_native_frei_lifestyle.csv")
lok_csv <- lies("ff2_moran_lokal.csv");     koh <- lies("ff2_moran_kohorten.csv")
nu <- lies("ff2_pol_groesse_null.csv")

ff2_trends <- rbind(
  cbind(mass = "eta2_fixed_voll",    tau_trend(co$jahr, co$eta2)),
  cbind(mass = "eta2_fixed_schnitt", tau_trend(re$jahr, re$eta2_fixed)),
  cbind(mass = "eta2_native",        tau_trend(re$jahr, re$eta2_native)),
  cbind(mass = "omega2_native",      tau_trend(re$jahr, re$omega2_native)),
  # steigt mechanisch mit k: Niveau berichten, nicht tau
  cbind(mass = "benchmark_native",   tau_trend(re$jahr, re$bench_native)),

  cbind(mass = "eta2_native_lifestyle",      tau_trend(nf$jahr, nf$eta2_native_ls)),
  cbind(mass = "omega2_native_lifestyle",    tau_trend(nf$jahr, nf$omega2_native_ls)),
  cbind(mass = "benchmark_native_lifestyle", tau_trend(nf$jahr, nf$bench)),

  cbind(mass = "between_gew",   tau_trend(ze$jahr, ze$between_gew)),
  cbind(mass = "within_gew",    tau_trend(ze$jahr, ze$within_gew)),
  cbind(mass = "between_ungew", tau_trend(ze$jahr, ze$between_ungew)),
  cbind(mass = "within_ungew",  tau_trend(ze$jahr, ze$within_ungew)),

  cbind(mass = "moran_spdep_k50", tau_trend(mo$jahr, mo$moran_I)),
  cbind(mass = "moran_k15",       tau_trend(mk$jahr[mk$k == 15],  mk$moran_I[mk$k == 15])),
  cbind(mass = "moran_k50",       tau_trend(mk$jahr[mk$k == 50],  mk$moran_I[mk$k == 50])),
  cbind(mass = "moran_k100",      tau_trend(mk$jahr[mk$k == 100], mk$moran_I[mk$k == 100])),
  cbind(mass = "moran_panel_k50", tau_trend(mp$jahr, mp$moran_I)),
  cbind(mass = "rel_nachbar_sd",  tau_trend(rs$jahr, rs$rel_sd))
)

# Vier Zeilen aus dem Threshold-Sweep auf dem Lebensstil-Panel
th <- lies("ff2_moran_threshold_sweep.csv")
th <- th[th$raum == "nativ", ]
for (tt in sort(unique(th$tau))) {
  d <- th[th$tau == tt, ]
  ff2_trends <- rbind(ff2_trends,
    cbind(mass = sprintf("moran_threshold_tau%03d", round(tt * 100)),
          tau_trend(d$jahr, d$moran_I)))
}

ff2_trends <- rbind(ff2_trends,
  cbind(mass = "moran_lifestyle_k50", tau_trend(lok_csv$jahr, lok_csv$moran_global)),
  cbind(mass = "moran_altbestand",    tau_trend(lok_csv$jahr, lok_csv$moran_alt)),
  # Achtung: diese Gruppe altert mit, siehe die Kohortenrechnung
  cbind(mass = "moran_zustromgruppe", tau_trend(lok_csv$jahr, lok_csv$moran_neu)),

  cbind(mass = "kohorte_moran_absolut", tau_trend(koh$kohorte, koh$moran_kohorte)),
  cbind(mass = "kohorte_relativ_feld",
        tau_trend(koh$kohorte, koh$moran_kohorte / koh$moran_global)),

  cbind(mass = "between_gew_null_perm", tau_trend(nu$jahr, nu$var_gew_null)),
  # Nicht-Befunde: die Größen-These ist am Nullmodell gescheitert
  cbind(mass = "ns_ueberschuss_rho",  tau_trend(nu$jahr, nu$ueberschuss_rho)),
  cbind(mass = "ns_ueberschuss_verh", tau_trend(nu$jahr, nu$ueberschuss_verh)))

print(ff2_trends, row.names = FALSE)
write_csv(ff2_trends, file.path(OUT_DIR, "ff2_trends.csv"))

                       mass    tau        p        start         ende n_jahre
            eta2_fixed_voll  0.667 1.27e-02  0.156853712  0.218955046       9
         eta2_fixed_schnitt  0.722 5.89e-03  0.156853712  0.247820385       9
                eta2_native  0.944 4.96e-05  0.156853712  0.347128442       9
              omega2_native  0.944 4.96e-05  0.149534716  0.332198915       9
           benchmark_native  0.944 4.96e-05  0.008592322  0.022318299       9
      eta2_native_lifestyle  1.000 5.51e-06  0.156853712  0.351316967       9
    omega2_native_lifestyle  1.000 5.51e-06  0.149534716  0.344560178       9
 benchmark_native_lifestyle  0.778 2.43e-03  0.008592322  0.010291152       9
                between_gew  0.722 5.89e-03  0.136093976  0.215362672       9
                 within_gew  0.167 6.12e-01  0.731555088  0.768230426       9
              between_ungew  0.444 1.19e-01  0.246879395  0.332946102       9
               within_ungew -0.111 7.61e-01  0.692746186  0.7303

In [33]:
cm <- cluster_jahr |> transmute(jahr, cluster, mittel = mean_z, n_subs = n)
cat(sprintf("Cluster-Mittelwerte: %d Zeilen, %d Cluster x %d Jahre\n",
            nrow(cm), n_distinct(cm$cluster), n_distinct(cm$jahr)))

write_csv(cm, file.path(OUT_DIR, "ff2_cluster_mittel.csv"))

Cluster-Mittelwerte: 855 Zeilen, 95 Cluster x 9 Jahre


In [34]:
status <- case_when(
  !leer & etikett == "false"                    ~ "A 2016er Urteil, nicht politisch -> drin",
  !leer & etikett == "true"                     ~ "B 2016er Urteil, politisch -> raus",
   leer & !is.na(etikett) & etikett == "false"  ~ "C imputiert, nicht politisch -> drin",
   leer & !is.na(etikett) & etikett == "true"   ~ "D imputiert, politisch -> raus",
  TRUE                                          ~ "E uneindeutig oder nie gelabelt -> raus")
names(status) <- names(etikett)

sc_wide <- read.csv(SCORE_CSV, check.names = FALSE)
names(sc_wide)[1] <- "subreddit"
S <- as.matrix(sc_wide[, paste0("score_", JAHRE)])
rownames(S) <- sc_wide$subreddit

st <- status[match(sc_wide$subreddit, names(status))]
st[is.na(st)] <- "F kein LLM-Eintrag -> raus"

cat("gescorte Subreddits je Jahr:\n")
print(setNames(as.integer(colSums(!is.na(S))), JAHRE))
cat("\nVereinigung ueber alle neun Jahre:", sum(rowSums(!is.na(S)) > 0), "Subreddits\n")

zeig <- function(v, titel) {
  tb <- table(v)
  df <- data.frame(status = names(tb), n = as.integer(tb),
                   anteil_prozent = round(100 * as.integer(tb) / sum(tb), 2),
                   row.names = NULL)
  df <- df[order(df$status), ]
  cat("\n== ", titel, "  (N = ", sum(tb), ")\n", sep = "")
  print(df, row.names = FALSE)
  invisible(df)
}

d_union <- zeig(st[rowSums(!is.na(S)) > 0], "Vereinigung über alle Jahre")
d_2016  <- zeig(st[!is.na(S[, "score_2016"])], "gescorte Landschaft 2016")
d_2024  <- zeig(st[!is.na(S[, "score_2024"])], "gescorte Landschaft 2024")

fix_alle <- read.csv(CLUSTER_CSV) |> filter(jahr == 2016, cluster != -1)
st_fix <- status[match(fix_alle$subreddit, names(status))]
st_fix[is.na(st_fix)] <- "F kein LLM-Eintrag -> raus"
d_fix <- zeig(st_fix, "fixe Partition 2016")

herkunft <- rbind(cbind(basis = "vereinigung_alle_jahre", d_union),
                  cbind(basis = "landschaft_2016",        d_2016),
                  cbind(basis = "landschaft_2024",        d_2024),
                  cbind(basis = "fixe_partition_2016",    d_fix))
write_csv(herkunft, file.path(OUT_DIR, "populationen_labelherkunft.csv"))

gescorte Subreddits je Jahr:
 2016  2017  2018  2019  2020  2021  2022  2023  2024 
16618 18426 20255 22409 24842 26930 28640 29446 28848 

Vereinigung ueber alle neun Jahre: 31284 Subreddits

== Vereinigung über alle Jahre  (N = 31284)
                                   status     n anteil_prozent
 A 2016er Urteil, nicht politisch -> drin 13678          43.72
       B 2016er Urteil, politisch -> raus   978           3.13
     C imputiert, nicht politisch -> drin 14251          45.55
           D imputiert, politisch -> raus   600           1.92
  E uneindeutig oder nie gelabelt -> raus  1514           4.84
               F kein LLM-Eintrag -> raus   263           0.84

== gescorte Landschaft 2016  (N = 16618)
                                   status     n anteil_prozent
 A 2016er Urteil, nicht politisch -> drin 13647          82.12
       B 2016er Urteil, politisch -> raus   976           5.87
     C imputiert, nicht politisch -> drin  1651           9.94
           D imputiert, poli

In [35]:
sessionInfo()

R version 4.5.1 (2025-06-13 ucrt)
Platform: x86_64-w64-mingw32/x64
Running under: Windows 11 x64 (build 26200)

Matrix products: default
  LAPACK version 3.12.1

locale:
[1] LC_COLLATE=German_Germany.utf8  LC_CTYPE=German_Germany.utf8   
[3] LC_MONETARY=German_Germany.utf8 LC_NUMERIC=C                   
[5] LC_TIME=German_Germany.utf8    

time zone: Europe/Berlin
tzcode source: internal

attached base packages:
[1] stats     graphics  grDevices utils     datasets  methods   base     

other attached packages:
 [1] spdep_1.4-2       sf_1.1-1          spData_2.3.5      lubridate_1.9.4  
 [5] forcats_1.0.0     stringr_1.5.1     dplyr_1.1.4       purrr_1.1.0      
 [9] readr_2.1.5       tidyr_1.3.1       tibble_3.3.0      ggplot2_3.5.2    
[13] tidyverse_2.0.0   dbscan_1.2.2      ineq_0.2-13       igraph_2.3.2     
[17] data.table_1.17.8

loaded via a namespace (and not attached):
 [1] s2_1.1.11          generics_0.1.4     class_7.3-23       KernSmooth_2.23-26
 [5] stringi_1.8.7      lat